# Évaluer avant de livrer — un protocole d'évaluation reproductible

**Notebook** : Evaluation-Livraison-01
**Série** : Evaluation-Livraison (projet cross-séries)
**Durée** : ~1 h
**Prérequis** : Python, lecture d'un fichier YAML, notions de métriques (exactitude, percentile)

## Objectifs d'apprentissage

- **Déclarer** avant toute mesure : la route du candidat, la référence, le juge et les seuils de décision.
- **Distinguer** une mesure (exactitude, abstention, latence, coût en jetons) d'une impression : le verdict est une comparaison à des seuils écrits, jamais une appréciation.
- **Rendre** une évaluation reproductible : la référence est lue et jamais modifiée, le juge est déterministe, les exclusions sont **affichées** plutôt que silencieuses.
- **Compter** le coût d'une réponse, pas seulement sa justesse : jetons par requête, jetons par bonne réponse.

## Navigation

- [Le terrain : trois artefacts](#le-terrain--trois-artefacts)
- [La route déclarée](#la-route-déclarée)
- [La banque de référence](#la-banque-de-référence)
- [Le juge](#le-juge)
- [Les seuils](#les-seuils)
- [La passe de mesure](#la-passe-de-mesure)
- [Les métriques](#les-métriques)
- [Le verdict](#le-verdict)
- [Exercices](#exercices)

> **Position de ce carnet dans le projet.** Il ouvre le projet cross-séries
> `Evaluation-Livraison` : il évalue un système que le dépôt livre lui-même — la
> banque de questions publiée — avec un juge déterministe, la clé de correction.
> Le juge probabiliste et son garde-fou anti-biais sont une suite possible, pas
> un résultat de cette première branche.

## Le terrain : trois artefacts

Le protocole articule trois éléments distincts, chacun avec un rôle précis.

| Rôle | Artefact | Question posée |
|---|---|---|
| **Candidat** | Un service de modèle, à une route déclarée | Que sait produire le système livré ? |
| **Référence** | La banque de questions et sa clé de correction | Par rapport à quoi juge-t-on la réponse ? |
| **Juge** | Une fonction déterministe, écrite ici | Comment passe-t-on de la réponse au verdict ? |

Cette séparation est une discipline d'évaluation. Sans référence, une sortie
ne se juge pas ; sans juge explicite, le résultat dépend de l'intuition du
lecteur. Le protocole s'arrête donc si la route n'est pas disponible et
vérifie le juge avant d'interroger le candidat.

La première branche de l'exercice garde le juge **déterministe** : c'est la clé
du jeu de référence. Un deuxième modèle qui noterait les réponses est un juge
probabiliste ; son biais, son accord avec la clé et son comportement sur les
cas ambigus demanderaient un protocole distinct, à construire ensuite.

**La route réelle du service n'est jamais devinée.** Le carnet lit
`VLLM_BASE_URL` depuis l'environnement ou, à défaut, le trousseau local ; le
modèle candidat est lui aussi nommé et vérifié auprès de cette route avant la
passe.

In [1]:
'''Declaration de la route : le candidat que l'on va evaluer, nomme avant toute mesure.

Le service est exige, jamais devine. La route et la cle se lisent dans
l'environnement (VLLM_BASE_URL, VLLM_API_KEY), a defaut dans le trousseau
local du depot (fichier .env ignore par git). Aucune des deux n'est ecrite
dans le carnet.
'''

import hashlib
import json
import os
import time
import urllib.error
import urllib.request
from collections import Counter
from pathlib import Path

import yaml

# Les repertoires sont cherches depuis le repertoire courant : le carnet vit
# dans le depot, et l'execution se fait depuis son propre dossier.
MARQUEUR = Path('MyIA.AI.Notebooks') / 'cross-series' / 'qcm'
DOSSIER_PROJET = Path('MyIA.AI.Notebooks') / 'cross-series' / 'Evaluation-Livraison'


def racine_depot() -> Path:
    '''Rend le dossier racine du depot, cherche depuis le repertoire courant.'''
    candidats = [Path.cwd(), *Path.cwd().parents]
    for candidat in candidats:
        if (candidat / MARQUEUR).is_dir() and (candidat / DOSSIER_PROJET).is_dir():
            return candidat
    for candidat in candidats:
        if (candidat / MARQUEUR).is_dir():
            return candidat
    return Path.cwd()


RACINE = racine_depot()
CHEMIN_TRESORERIE = RACINE / 'MyIA.AI.Notebooks' / 'GenAI' / '.env'
try:
    CHEMIN_RELATIF = CHEMIN_TRESORERIE.relative_to(RACINE).as_posix()
except ValueError:
    CHEMIN_RELATIF = 'MyIA.AI.Notebooks/GenAI/.env'


def lire_trousseau(chemin: Path, nom: str) -> str | None:
    '''Extrait une variable du fichier .env local ; None si le fichier ou la variable est absent.'''
    if not chemin.is_file():
        return None
    contenu = chemin.read_text(encoding='utf-8', errors='replace')
    for ligne in contenu.splitlines():
        if '=' not in ligne:
            continue
        cle, _, valeur = ligne.partition('=')
        if cle.strip() == nom:
            return valeur.strip().strip(chr(34)).strip(chr(39))
    return None


def variable(nom: str) -> str:
    '''Lit une variable dans l environnement, a defaut dans le trousseau local du depot.'''
    return os.environ.get(nom, '') or lire_trousseau(CHEMIN_TRESORERIE, nom) or ''


# Le candidat evalue est declare par son identifiant servi, puis verifie parmi
# les modeles que la route sert. Un candidat remplace par un autre n'est plus
# le meme candidat : le verdict ne lui serait pas attribuable.
CANDIDAT_DECLARE = variable('VLLM_MODELE') or 'qwen3.6-35b-a3b'
CLE = variable('VLLM_API_KEY')
ROUTE = variable('VLLM_BASE_URL').rstrip('/')

# Sans route ni cle, le carnet s arrete ici. Un repli qui rendrait des
# abstentions remplacerait la mesure par du silence : il n y en a pas. Les
# routes du siege et de la facade distante sont documentees dans
# docs/genai/model-mapping-openai.md.
if not ROUTE:
    raise RuntimeError(
        'VLLM_BASE_URL est absente. Ce carnet interroge un service de chat '
        'compatible OpenAI et ne peut pas deviner son adresse : exporter la '
        'route du service dans VLLM_BASE_URL (sur le siege ai-01 : '
        'http://127.0.0.1:5002/v1), puis relancer. Un environnement absent se '
        'repare, il ne se contourne pas.'
    )
if not CLE:
    raise RuntimeError(
        'VLLM_API_KEY est absente. La cle du service se lit dans la variable '
        'd environnement VLLM_API_KEY ou dans le trousseau local du depot ('
        + CHEMIN_RELATIF
        + '), jamais dans le carnet. Provisionner la cle, puis relancer.'
    )


def lister_modeles(base: str, cle: str, delai: int = 8) -> list[str]:
    '''Interroge le point /models et rend les identifiants de modeles servis.'''
    requete = urllib.request.Request(
        base + '/models', headers={'Authorization': 'Bearer ' + cle}
    )
    try:
        with urllib.request.urlopen(requete, timeout=delai) as reponse:
            charge = json.loads(reponse.read().decode('utf-8', errors='replace'))
    except (urllib.error.URLError, OSError, ValueError) as incident:
        # Une route injoignable s affiche avec sa cause ; elle ne rend pas une liste vide.
        raise RuntimeError(
            'la route declaree ne repond pas (' + base + ') : '
            + type(incident).__name__ + ' ' + str(incident)
            + '. Verifier que le service de chat est lance, puis relancer.'
        ) from incident
    return [m.get('id', '') for m in charge.get('data', []) if m.get('id')]


MODELES_SERVIS = lister_modeles(ROUTE, CLE)
if CANDIDAT_DECLARE not in MODELES_SERVIS:
    raise RuntimeError(
        'le candidat declare (' + CANDIDAT_DECLARE + ') n est pas servi par la '
        'route (' + ROUTE + ') ; modeles servis : ' + ', '.join(MODELES_SERVIS)
        + '. Le protocole ne remplace pas son candidat : corriger VLLM_MODELE '
        'ou la route, puis relancer.'
    )
MODELE_CANDIDAT = CANDIDAT_DECLARE

print('=== Route declaree ===')
print('fichier de tresorerie (chemin relatif au depot) :', CHEMIN_RELATIF)
print('cle VLLM :', 'presente')
print('route retenue :', ROUTE)
print('modeles servis :', ', '.join(MODELES_SERVIS))
print('candidat evalue :', MODELE_CANDIDAT)


=== Route declaree ===
fichier de tresorerie (chemin relatif au depot) : MyIA.AI.Notebooks/GenAI/.env
cle VLLM : presente
route retenue : http://127.0.0.1:5002/v1
modeles servis : qwen3.6-35b-a3b, swift-1.5-27b
candidat evalue : qwen3.6-35b-a3b


### Lecture du résultat

Trois choses sont déclarées avant la moindre question posée.

- La **route** est une adresse HTTP compatible avec l'API OpenAI, servie par le modèle local du siège. Elle est lue dans un ordre fixe : la variable d'environnement `VLLM_BASE_URL`, puis le trousseau local du dépôt. Elle est affichée dans les sorties, pour que la mesure soit rejouée contre la même adresse.
- Le **candidat** est le modèle que l'on interroge, nommé par son identifiant servi (`VLLM_MODELE`). Le carnet demande à la route la liste des modèles qu'elle sert et s'arrête si le candidat déclaré n'en fait pas partie : un candidat remplacé par un autre n'est plus le même candidat, et le verdict ne lui serait pas attribuable.
- La **clé** vient de l'environnement (`VLLM_API_KEY`), à défaut du trousseau local du dépôt. Elle n'apparaît ni dans le carnet ni dans les sorties : le protocole ne rend compte que de la mesure.

Le service est exigé, jamais deviné. Sans route, sans clé, ou face à une route
qui ne répond pas, le carnet s'arrête à cette cellule, avec un message qui nomme
ce qui manque. Les routes du siège et de la façade distante sont documentées
dans `docs/genai/model-mapping-openai.md`. Un repli qui rendrait des
abstentions remplacerait la mesure par du silence : un environnement absent se
répare, il ne se contourne pas.

In [2]:
'''Chargement de la banque publiee et recensement par theme et par type.

La banque est lue, jamais modifiee : c'est la reference du protocole. Elle est
en outre epinglee : son empreinte est comparee a celle du manifeste du projet.
'''

DOSSIER_QCM = RACINE / 'MyIA.AI.Notebooks' / 'cross-series' / 'qcm'
CHEMIN_MANIFESTE = RACINE / DOSSIER_PROJET / 'evaluation-livraison.json'
CHEMINS_QCM = sorted(DOSSIER_QCM.glob('*.yaml'))


def empreinte_banque(chemins: list[Path]) -> str:
    '''Empreinte SHA-256 des fichiers de la banque, pris dans l ordre de leur nom.

    L empreinte porte sur le texte, pas sur les octets du poste : fins de ligne
    ramenees a LF et marque d encodage retiree, pour qu un meme fichier garde
    la meme empreinte d une machine a l autre.
    '''
    hacheur = hashlib.sha256()
    for chemin in chemins:
        texte = chemin.read_bytes().lstrip(b'\xef\xbb\xbf').replace(b'\r\n', b'\n')
        hacheur.update(chemin.name.encode('utf-8') + b'\n')
        hacheur.update(hashlib.sha256(texte).hexdigest().encode('ascii') + b'\n')
    return hacheur.hexdigest()


# Chaque question garde le nom de son support : la provenance reste lisible
# jusque dans les exclusions affichees plus bas.
BANQUE: list[dict] = []
for chemin in CHEMINS_QCM:
    questions = yaml.safe_load(chemin.read_text(encoding='utf-8')) or []
    for question in questions:
        question['_fichier'] = chemin.name
        BANQUE.append(question)

EMPREINTE_BANQUE = empreinte_banque(CHEMINS_QCM)
REFERENCE_MANIFESTE = json.loads(CHEMIN_MANIFESTE.read_text(encoding='utf-8')).get('reference') or {}
EMPREINTE_EPINGLEE = REFERENCE_MANIFESTE.get('empreinte_sha256', '')

# Le verdict consigne dans le depot decrit une banque precise. Si la banque
# change, il ne la decrit plus : le carnet s arrete plutot que de laisser lire
# un chiffre perime a cote d une reference qui a bouge.
if EMPREINTE_BANQUE != EMPREINTE_EPINGLEE:
    raise RuntimeError(
        'la banque de questions a change depuis l epinglage du manifeste. '
        'Empreinte mesuree : ' + EMPREINTE_BANQUE + ' ; empreinte epinglee : '
        + (EMPREINTE_EPINGLEE or 'absente') + '. Relancer la passe de mesure, '
        'puis mettre a jour reference.empreinte_sha256 dans le manifeste.'
    )

print('=== Banque publiee ===')
print('questions publiees :', len(BANQUE))
print('fichiers :', len(CHEMINS_QCM))
print('empreinte SHA-256 de la banque :', EMPREINTE_BANQUE)
print('empreinte epinglee dans le manifeste : identique')
print('par type :', dict(sorted(Counter(q['type'] for q in BANQUE).items())))
print('par theme :')
for theme, effectif in sorted(Counter(q['theme'] for q in BANQUE).items()):
    print('  -', theme, ':', effectif)


=== Banque publiee ===
questions publiees : 146
fichiers : 6
empreinte SHA-256 de la banque : ae4434d3edc94537016514dfbf632ae79ec961db813be1ece341ea077db17daf
empreinte epinglee dans le manifeste : identique
par type : {'matching': 1, 'multichoice': 131, 'truefalse': 14}
par theme :
  - dl-evaluation-seance : 32
  - ia-1-introduction-agents : 15
  - ia-2-resolution-problemes : 35
  - ia-3-logique-bases-connaissances : 19
  - ia-4-systemes-probabilistes : 24
  - ia-5-apprentissage : 21


### Lecture du résultat

La banque publiée est un **actif de référence** : c'est elle qui sera jugée, et
le protocole ne la modifiera pas. Le recensement sert à deux choses.

- Le **volume par thème** fixe ce qu'on peut conclure. Un thème peu peuplé donne
  une exactitude très bruitée : quelques questions suffisent à la faire bouger
  de plusieurs points.
- La **répartition par type** dit ce que le juge sait noter. Une question à
  réponse unique se note par une lettre ; une question à appariements se note
  autrement, et une question qui admet plusieurs réponses n'est pas notable par
  une lettre.

C'est le second point qui commande la suite : le jeu de référence n'est pas la
banque entière.

La banque est en outre **épinglée**. Le carnet calcule une empreinte SHA-256 de
ses fichiers et la compare à celle que le manifeste du projet
(`evaluation-livraison.json`) a inscrite. L'empreinte porte sur le texte des
fichiers, fins de ligne ramenées à LF et marque d'encodage retirée : un même
fichier garde la même empreinte d'une machine à l'autre. Si la banque change,
le carnet s'arrête — le verdict consigné dans le dépôt décrirait une banque qui
n'existe plus.

In [3]:
'''Jeu de reference : ce que le juge a lettre sait noter.

Deux familles sont ecartees, et l'ecartement est affiche :
- le type `matching`, ou la reponse attendue est un appariement et non une
  lettre ;
- les questions a reponses multiples, ou la lettre rendue n'identifie plus une
  reponse unique.
'''

TYPES_NOTABLES = ('multichoice', 'truefalse')


def lettre_de(question: dict) -> list[str]:
    '''Rend les lettres des options marquees correctes.'''
    return [
        chr(ord('A') + index)
        for index, option in enumerate(question.get('options', []))
        if option.get('correcte')
    ]


# Le tri se fait question par question, et chaque ecart garde son motif.
JEU_REFERENCE: list[dict] = []
EXCLUSIONS: list[tuple[str, str]] = []
for question in BANQUE:
    identifiant = question['id']
    if question.get('type') not in TYPES_NOTABLES:
        EXCLUSIONS.append((identifiant, 'type ' + str(question.get('type'))))
    elif question.get('choix_unique') is not True:
        EXCLUSIONS.append((identifiant, 'reponses multiples'))
    elif len(lettre_de(question)) != 1:
        EXCLUSIONS.append((identifiant, 'cle de correction absente ou multiple'))
    else:
        JEU_REFERENCE.append(question)

print('=== Jeu de reference ===')
print('questions publiees :', len(BANQUE))
print('questions notables par le juge a lettre :', len(JEU_REFERENCE))
print('questions ecartees :', len(EXCLUSIONS))
for identifiant, motif in EXCLUSIONS:
    print('  -', identifiant, ':', motif)
print('repartition du jeu de reference :')
for theme, effectif in sorted(Counter(q['theme'] for q in JEU_REFERENCE).items()):
    print('  -', theme, ':', effectif)
print('par type :', dict(sorted(Counter(q['type'] for q in JEU_REFERENCE).items())))


=== Jeu de reference ===
questions publiees : 146
questions notables par le juge a lettre : 135
questions ecartees : 11
  - ia1-001 : type matching
  - ia1-012 : reponses multiples
  - ia2-018 : reponses multiples
  - ia2-019 : reponses multiples
  - ia2-024 : reponses multiples
  - ia3-015 : reponses multiples
  - ia3-016 : reponses multiples
  - ia3-019 : reponses multiples
  - ia4-005 : reponses multiples
  - ia5-007 : reponses multiples
  - ia5-014 : reponses multiples
repartition du jeu de reference :
  - dl-evaluation-seance : 32
  - ia-1-introduction-agents : 13
  - ia-2-resolution-problemes : 32
  - ia-3-logique-bases-connaissances : 16
  - ia-4-systemes-probabilistes : 23
  - ia-5-apprentissage : 19
par type : {'multichoice': 121, 'truefalse': 14}


### Lecture du résultat

Le jeu de référence comprend les questions que le juge à lettre sait noter. Les
questions écartées sont nommées, avec le motif de l'écartement : cette liste est
une **mesure**, pas une convention silencieuse.

Deux raisons d'écarter, deux conséquences :

- **`matching`** : la réponse attendue est un appariement, une gauche et une
  droite. Un juge à lettre ne peut pas la noter sans changer de juge.
- **réponses multiples** : la question admet plusieurs bonnes réponses. La
  lettre rendue n'identifie alors plus une réponse unique, et la comparer à une
  seule clé pénaliserait une bonne réponse.

Écarter ces questions ne rend pas le protocole plus flatteur : cela rend la
mesure **interprétable**. Le taux d'exactitude porte alors sur des questions
dont la clé est unique, et le rapport peut dire sans ambiguïté ce qu'il mesure.


In [4]:
'''Le juge : deterministe. Il extrait la lettre rendue et la compare a la cle.

Trois verdicts possibles : `correct`, `incorrect`, `abstention`. Une reponse
inexploitable (pas de lettre) est une abstention, pas une faute.
'''

# Tout separateur est une frontiere de mot : la ponctuation, les retours a la
# ligne, et les marqueurs Markdown dont une reponse habille souvent sa lettre
# (gras, code, tiret de liste). Le backtick s ecrit chr(96) pour rester lisible.
SEPARATEURS = ' .,;:!?()[]{}/-*|~=<>' + chr(96) + chr(34) + chr(39) + chr(9) + chr(10) + chr(13)


def mots_de(reponse: str | None) -> list[str]:
    '''Decoupe la reponse en mots ; tout separateur est une frontiere.'''
    mots: list[str] = []
    courant = ''
    for caractere in reponse or '':
        if caractere in SEPARATEURS:
            mots.append(courant)
            courant = ''
        else:
            courant += caractere
    mots.append(courant)
    return mots


def lettres_isolees(reponse: str | None) -> list[str]:
    '''Rend, dans l ordre et sans doublon, les lettres isolees de la reponse.

    Une lettre isolee est un mot d une seule lettre alphabetique. Une phrase
    d explication separee de sa lettre est donc lue correctement, et une phrase
    sans lettre isolee ne vaut aucune reponse.
    '''
    lettres: list[str] = []
    for mot in mots_de(reponse):
        if len(mot) == 1 and mot.isalpha() and mot.upper() not in lettres:
            lettres.append(mot.upper())
    return lettres


def extraire_lettre(reponse: str | None) -> str | None:
    '''Rend la lettre rendue : la premiere lettre isolee de la reponse, ou None.'''
    lettres = lettres_isolees(reponse)
    return lettres[0] if lettres else None


def juger(reponse: str | None, question: dict) -> str:
    '''Rend le verdict d'une reponse pour une question du jeu de reference.'''
    lettre = extraire_lettre(reponse)
    if lettre is None:
        return 'abstention'
    nb_options = len(question.get('options', []))
    if not (0 <= ord(lettre) - ord('A') < nb_options):
        return 'abstention'
    return 'correct' if lettre in lettre_de(question) else 'incorrect'


# Auto-tests du juge : ils tournent avant que le candidat ne soit interroge.
# La question de test porte trois options : la lettre D est hors de la plage,
# la lettre C est dans la plage mais fausse -- les deux branches doivent se
# distinguer, sinon le juge confond une faute et une reponse inexploitable.
# Les lettres habillees (gras, code, tiret) doivent se lire comme la lettre nue.
QUESTION_TEST = {
    'options': [
        {'texte': 'premiere proposition', 'correcte': False},
        {'texte': 'deuxieme proposition', 'correcte': True},
        {'texte': 'troisieme proposition', 'correcte': False},
    ]
}
CAS_DE_TEST = [
    ('B', 'correct'),
    ('La reponse correcte est B.', 'correct'),
    ('b', 'correct'),
    ('**B**', 'correct'),
    ('- `B`', 'correct'),
    ('**Reponse : B**', 'correct'),
    ('C', 'incorrect'),
    ('La reponse est C, pas B.', 'incorrect'),
    ('D', 'abstention'),
    ('Je ne sais pas.', 'abstention'),
    ('', 'abstention'),
]
reussis = 0
for reponse, statut_attendu in CAS_DE_TEST:
    verdict = juger(reponse, QUESTION_TEST)
    conforme = verdict == statut_attendu
    reussis += 1 if conforme else 0
    print('   ', repr(reponse), '->', verdict, '| attendu', statut_attendu, '|',
          'conforme' if conforme else 'ECART')
print('auto-tests du juge :', reussis, '/', len(CAS_DE_TEST))


    'B' -> correct | attendu correct | conforme
    'La reponse correcte est B.' -> correct | attendu correct | conforme
    'b' -> correct | attendu correct | conforme
    '**B**' -> correct | attendu correct | conforme
    '- `B`' -> correct | attendu correct | conforme
    '**Reponse : B**' -> correct | attendu correct | conforme
    'C' -> incorrect | attendu incorrect | conforme
    'La reponse est C, pas B.' -> incorrect | attendu incorrect | conforme
    'D' -> abstention | attendu abstention | conforme
    'Je ne sais pas.' -> abstention | attendu abstention | conforme
    '' -> abstention | attendu abstention | conforme
auto-tests du juge : 11 / 11


### Lecture du résultat

Le juge tient en trois verdicts, et la distinction entre **faute** et **absence
de réponse** est la plus utile d'un protocole d'évaluation.

- `correct` : la lettre rendue est celle de la clé.
- `incorrect` : une lettre exploitable a été rendue, elle n'est pas celle de la clé.
- `abstention` : la réponse ne porte pas de lettre utilisable — réponse vide, phrase sans lettre, ou lettre hors de la plage des options proposées.

La lettre rendue est la **première lettre isolée** de la réponse : un mot d'une
seule lettre, entouré de frontières de mot. Tout séparateur est une frontière —
la ponctuation, les retours à la ligne et les marqueurs Markdown dont un modèle
habille souvent sa lettre (`**B**`, `` `B` ``, un tiret de liste). Une phrase
d'explication suivie de sa lettre se lit donc correctement, et une phrase sans
lettre isolée ne vaut aucune réponse. Ce que cette règle a à trancher se mesure
plus bas : une réponse qui porte plusieurs lettres isolées dans la plage des
options est ambiguë pour un juge à lettre, et le détail des échecs en affiche le
nombre.

Les auto-tests affichés plus haut vérifient ces trois branches et le traitement
des lettres habillées sur des cas connus, **avant** que le candidat ne soit
interrogé : un juge que l'on ne vérifie pas est un juge dont les erreurs se
confondent avec celles du candidat.

Le juge est ici **déterministe** : la même réponse rend toujours le même
verdict. Un juge probabiliste — un second modèle qui note les réponses — est
une extension à construire, avec un garde-fou anti-bias ; il n'est pas employé
ici.

## Les seuils

Un seuil est une condition écrite **avant** la mesure. Un seuil écrit après ne
décide plus : il décrit ce qui s'est produit, et l'on peut alors lui faire dire
n'importe quoi.

Les seuils ci-dessous valent pour ce protocole, sur cette banque de référence,
avec ce candidat. Ils sont modifiables — mais ils sont fixés ici, une fois, et
le verdict de la fin du carnet en tient compte sans jamais les réécrire.


In [5]:
'''Les seuils de decision : ecrits avant la passe de mesure.

Un seuil ecrit apres la mesure ne decide rien : il decrit ce qui s'est produit.
'''

# Les six conditions qui rendent le verdict du protocole, et rien d'autre.
SEUILS = {
    'exactitude_globale_min': 0.50,
    'exactitude_par_theme_min': 0.35,
    'taux_abstention_max': 0.10,
    'latence_p95_s_max': 30.0,
    'jetons_par_requete_max': 600,
    'jetons_par_bonne_reponse_max': 4000,
}

print('=== Seuils declares avant la mesure ===')
for nom, valeur in SEUILS.items():
    print('  -', nom, ':', valeur)


=== Seuils declares avant la mesure ===
  - exactitude_globale_min : 0.5
  - exactitude_par_theme_min : 0.35
  - taux_abstention_max : 0.1
  - latence_p95_s_max : 30.0
  - jetons_par_requete_max : 600
  - jetons_par_bonne_reponse_max : 4000


### Lecture du résultat

Le verdict du protocole repose sur **six seuils** : un global, un par thème, un
d'abstention, un de latence et deux de coût. Ils sont écrits avant la passe et
restent fixes quelle que soit la sortie obtenue.

- L'**exactitude globale** demande un résultat utilisable sur l'ensemble du jeu.
- L'**exactitude par thème** empêche une bonne moyenne de masquer une famille
  de questions mal traitée.
- Le **taux d'abstention** borne la part de réponses que le juge n'a pas su
  noter.
- La **latence au percentile haut** borne le temps de réponse vécu par une
  requête lente, sans laisser la médiane masquer les attentes longues.
- Les **jetons par requête** mesurent le coût courant ; les **jetons par bonne
  réponse** le coût d'une réponse utile.

Chaque seuil se lit indépendamment, puis le verdict les agrège : il n'y a pas de
compensation entre un seuil largement franchi et un autre manqué. Le carnet
imprime les seuils déclarés avant de poser la première question ; la cellule de
verdict les confronte ensuite aux mesures.

### Interroger le candidat

La fonction d'appel est volontairement **générique** : elle ne connaît ni la
question, ni le juge, ni les seuils. Elle reçoit une question, rend une trace
— réponse lisible, latence, jetons, incident éventuel — et laisse le jugement
à d'autres maillons. C'est ce qui permet de changer de juge, ou de route, sans
toucher à la mesure.

Deux garde-fous y sont inscrits : le raisonnement est désactivé et la
température est nulle, parce qu'on veut une lettre reproductible et non un
raisonnement ; et une panne de transport n'interrompt pas la passe, elle est
comptée comme une abstention.

Les seuils, eux, sont déjà écrits. C'est le point : au moment où le candidat
répond, la condition de réussite est connue et ne peut plus être ajustée.


In [6]:
'''L'appel au candidat : une question, une reponse, une trace de requete.

Chaque requete est tracee : latence, jetons consommes, incident eventuel. Une
panne de transport n'interrompt pas la passe : elle est comptee comme
abstention.
'''


def interroger(question: dict, delai: int = 120) -> dict:
    '''Interroge le candidat sur une question et rend la trace de la requete.'''
    options = [option['texte'] for option in question.get('options', [])]
    lettres = [chr(ord('A') + index) for index in range(len(options))]
    corps = chr(10).join(
        '%s. %s' % (lettre, texte) for lettre, texte in zip(lettres, options)
    )
    invite = chr(10).join([
        'Reponds par UNE SEULE lettre parmi %s. Ne justifie pas.' % ', '.join(lettres),
        '',
        question['enonce'],
        '',
        corps,
        '',
        'Reponse :',
    ])
    # Le raisonnement est desactive et la temperature est nulle : on veut une
    # lettre reproductible, pas un raisonnement.
    charge = {
        'model': MODELE_CANDIDAT,
        'messages': [{'role': 'user', 'content': invite}],
        'temperature': 0,
        'max_tokens': 64,
        'chat_template_kwargs': {'enable_thinking': False},
    }
    depart = time.perf_counter()
    if not ROUTE or not MODELE_CANDIDAT:
        return {
            'id': question['id'],
            'reponse': None,
            'latence_s': 0.0,
            'jetons': 0,
            'incident': 'route absente',
        }
    try:
        requete = urllib.request.Request(
            ROUTE + '/chat/completions',
            data=json.dumps(charge).encode('utf-8'),
            method='POST',
            headers={
                'Authorization': 'Bearer ' + (CLE or ''),
                'Content-Type': 'application/json',
            },
        )
        with urllib.request.urlopen(requete, timeout=delai) as reponse:
            charge_recu = json.loads(reponse.read().decode('utf-8', errors='replace'))
    except (urllib.error.URLError, OSError, ValueError) as incident:
        return {
            'id': question['id'],
            'reponse': None,
            'latence_s': time.perf_counter() - depart,
            'jetons': 0,
            'incident': type(incident).__name__,
        }
    choix = (charge_recu.get('choices') or [{}])[0]
    contenu = (choix.get('message') or {}).get('content')
    usage = charge_recu.get('usage') or {}
    return {
        'id': question['id'],
        'reponse': contenu,
        'latence_s': time.perf_counter() - depart,
        'jetons': int(usage.get('total_tokens') or 0),
        'jetons_entree': int(usage.get('prompt_tokens') or 0),
        'jetons_sortie': int(usage.get('completion_tokens') or 0),
        'incident': None,
    }


### Lecture du résultat

Deux paramètres fixent le comportement du candidat : la température est
réglée à zéro et le raisonnement est désactivé (on veut une lettre, pas une
justification). Les deux sont déclarés dans la requête, donc auditables.

La trace de chaque requête est conservée : la latence, les jetons consommés, et
l'incident éventuel. Une panne de transport n'interrompt pas la passe : elle
est comptée comme une abstention, et le rapport le dit.

## La passe de mesure

Le protocole interroge le candidat sur **chaque** question du jeu de référence,
une question par requête, dans l'ordre de la banque. Le passage suivant affiche
la progression, pour qu'une passe longue reste lisible en cours d'exécution.

In [7]:
'''La passe : le candidat repond a chaque question du jeu de reference.'''

RESULTATS: list[dict] = []
for position, question in enumerate(JEU_REFERENCE, start=1):
    trace = interroger(question)
    trace['theme'] = question['theme']
    trace['type'] = question['type']
    # Le nombre d'options est conserve : il sert de reference de hasard.
    trace['nb_options'] = len(question.get('options', []))
    trace['statut'] = juger(trace['reponse'], question)
    RESULTATS.append(trace)
    # Un point de progression tous les vingt-cinq appels : la passe reste
    # lisible sans noyer la sortie sous une ligne par requete.
    if position % 25 == 0:
        print('progression :', position, '/', len(JEU_REFERENCE))

print('=== Passe terminee ===')
print('requetes emises :', len(RESULTATS))
print('reponses exploitables :', sum(1 for r in RESULTATS if r['statut'] != 'abstention'))
print('abstentions :', sum(1 for r in RESULTATS if r['statut'] == 'abstention'))
print('incidents de transport :', sum(1 for r in RESULTATS if r['incident']))
print('jetons consommes au total :', sum(r['jetons'] for r in RESULTATS))


progression : 25 / 135


progression : 50 / 135


progression : 75 / 135


progression : 100 / 135


progression : 125 / 135


=== Passe terminee ===
requetes emises : 135
reponses exploitables : 135
abstentions : 0
incidents de transport : 0
jetons consommes au total : 20638


### Lecture du résultat

La passe a posé **une requête par question du jeu de référence**. Trois compteurs
méritent d'être lus ensemble.

- Les **réponses exploitables** et les **abstentions** se complètent : le total
  est le nombre de questions du jeu. Une passe sans réponse exploitable ne
  mesurerait pas le candidat.
- Les **incidents de transport** sont comptés séparément des abstentions. Une
  panne n'est pas une réponse du candidat ; elle ne doit pas disparaître dans
  le score.
- Le **total de jetons** alimente les métriques de coût par requête et par
  bonne réponse.

La sortie affiche ces nombres pour chaque passe. Le manifeste consigne les
mesures de référence, tandis que les seuils restent ceux déclarés avant la
première question.

In [8]:
'''Les metriques : justesse, abstention, latence, cout.'''


def percentile(valeurs: list[float], part: float) -> float:
    '''Percentile lineaire d'une liste de valeurs.'''
    if not valeurs:
        return 0.0
    triees = sorted(valeurs)
    position = part * (len(triees) - 1)
    bas = int(position)
    haut = min(bas + 1, len(triees) - 1)
    return triees[bas] + (position - bas) * (triees[haut] - triees[bas])


# Les compteurs sont construits une fois, puis reutilises par le verdict.
total = len(RESULTATS)
correctes = sum(1 for r in RESULTATS if r['statut'] == 'correct')
abstentions = sum(1 for r in RESULTATS if r['statut'] == 'abstention')
incorrectes = total - correctes - abstentions
latences = [r['latence_s'] for r in RESULTATS]
jetons_total = sum(r['jetons'] for r in RESULTATS)

METRIQUES = {
    'questions': total,
    'correctes': correctes,
    'incorrectes': incorrectes,
    'abstentions': abstentions,
    'exactitude_globale': (correctes / total) if total else 0.0,
    'taux_abstention': (abstentions / total) if total else 0.0,
    'latence_p50_s': percentile(latences, 0.50),
    'latence_p95_s': percentile(latences, 0.95),
    'jetons_total': jetons_total,
    'jetons_par_requete': (jetons_total / total) if total else 0.0,
    'jetons_par_bonne_reponse': (jetons_total / correctes) if correctes else 0.0,
}


def agreger_par_cle(resultats: list[dict], cle: str) -> dict[str, dict]:
    '''Regroupe les traces par une cle et calcule l'exactitude de chaque groupe.'''
    groupes: dict[str, dict] = {}
    for trace in resultats:
        nom = trace[cle]
        groupe = groupes.setdefault(nom, {'questions': 0, 'correctes': 0})
        groupe['questions'] += 1
        if trace['statut'] == 'correct':
            groupe['correctes'] += 1
    for groupe in groupes.values():
        groupe['exactitude'] = (
            groupe['correctes'] / groupe['questions'] if groupe['questions'] else 0.0
        )
    return groupes


METRIQUES_PAR_THEME = agreger_par_cle(RESULTATS, 'theme')
METRIQUES_PAR_TYPE = agreger_par_cle(RESULTATS, 'type')

# Le hasard de reference : pour chaque question, une option tiree au hasard.
# Une exactitude ne se lit qu'en face de lui, sinon un score faible ne dit pas
# si le candidat a repondu faux ou si la question etait devinee.
HASARD_PAR_THEME: dict[str, float] = {}
for theme in METRIQUES_PAR_THEME:
    tirages = [
        1.0 / trace['nb_options']
        for trace in RESULTATS
        if trace['theme'] == theme and trace['nb_options']
    ]
    HASARD_PAR_THEME[theme] = (sum(tirages) / len(tirages)) if tirages else 0.0
tirages_global = [
    1.0 / trace['nb_options'] for trace in RESULTATS if trace['nb_options']
]
HASARD_GLOBAL = (sum(tirages_global) / len(tirages_global)) if tirages_global else 0.0

print('=== Exactitude ===')
print('exactitude globale : %.4f (%d bonnes reponses sur %d)' % (
    METRIQUES['exactitude_globale'], correctes, total))
print('taux d abstention : %.4f' % METRIQUES['taux_abstention'])
print('reponses incorrectes :', incorrectes)
print('=== Par theme ===')
for theme, valeurs in sorted(METRIQUES_PAR_THEME.items()):
    print('  -', theme, ': %.4f (%d/%d)' % (
        valeurs['exactitude'], valeurs['correctes'], valeurs['questions']))
print('=== Par type ===')
for type_question, valeurs in sorted(METRIQUES_PAR_TYPE.items()):
    print('  -', type_question, ': %.4f (%d/%d)' % (
        valeurs['exactitude'], valeurs['correctes'], valeurs['questions']))
print('=== Hasard de reference (une option tiree au hasard) ===')
print('global : %.4f' % HASARD_GLOBAL)
for theme in sorted(HASARD_PAR_THEME):
    hasard = HASARD_PAR_THEME[theme]
    ratio = METRIQUES_PAR_THEME[theme]['exactitude'] / hasard if hasard else 0.0
    print('  -', theme, ': hasard %.4f | exactitude %.4f | ratio %.2f' % (
        hasard, METRIQUES_PAR_THEME[theme]['exactitude'], ratio))
print('=== Latence (secondes) ===')
print('p50 : %.2f' % METRIQUES['latence_p50_s'])
print('p95 : %.2f' % METRIQUES['latence_p95_s'])
print('=== Cout (jetons) ===')
print('total :', jetons_total)
print('par requete : %.1f' % METRIQUES['jetons_par_requete'])
print('par bonne reponse : %.1f' % METRIQUES['jetons_par_bonne_reponse'])


=== Exactitude ===
exactitude globale : 0.6370 (86 bonnes reponses sur 135)
taux d abstention : 0.0000
reponses incorrectes : 49
=== Par theme ===
  - dl-evaluation-seance : 0.8125 (26/32)
  - ia-1-introduction-agents : 0.7692 (10/13)
  - ia-2-resolution-problemes : 0.3125 (10/32)
  - ia-3-logique-bases-connaissances : 0.6875 (11/16)
  - ia-4-systemes-probabilistes : 0.5217 (12/23)
  - ia-5-apprentissage : 0.8947 (17/19)
=== Par type ===
  - multichoice : 0.6116 (74/121)
  - truefalse : 0.8571 (12/14)
=== Hasard de reference (une option tiree au hasard) ===
global : 0.2005
  - dl-evaluation-seance : hasard 0.3620 | exactitude 0.8125 | ratio 2.24
  - ia-1-introduction-agents : hasard 0.1586 | exactitude 0.7692 | ratio 4.85
  - ia-2-resolution-problemes : hasard 0.1225 | exactitude 0.3125 | ratio 2.55
  - ia-3-logique-bases-connaissances : hasard 0.1740 | exactitude 0.6875 | ratio 3.95
  - ia-4-systemes-probabilistes : hasard 0.1427 | exactitude 0.5217 | ratio 3.66
  - ia-5-apprentissage

### Lecture du résultat

Les métriques se lisent en trois temps.

- **L'exactitude globale** se lit dans la sortie qui précède : elle donne les
  bonnes réponses rapportées aux questions posées. Elle ne se compare qu'à une
  référence dont la clé est unique, prix du tri opéré sur la banque.
- Le **taux d'abstention** isole les cas où le protocole n'aurait pas su ce que
  le candidat répondait. Il est nul sur cette passe.
- **La latence et les jetons** disent ce que coûte une réponse. Les percentiles
  et les coûts unitaires sont affichés dans la sortie, au même endroit que
  leur règle de calcul.

La dispersion par thème est plus instructive que la moyenne : la sortie montre
l'écart entre les thèmes les plus faibles et les plus forts. Un candidat qui
réussit « en moyenne » peut masquer un angle mort complet sur une famille de
questions — c'est tout l'intérêt d'une mesure par thème.

Aucune de ces exactitudes n'a de sens isolée. Le **hasard de référence** — une
option tirée au hasard — est calculé à partir de la taille du choix pour chaque
question. Un score se lit toujours en face de ce repère : c'est lui qui
distingue « le candidat s'est trompé » de « la question était devinable ».

Le coût par bonne réponse est la métrique la plus parlante pour un budget :
elle divise la dépense totale par le nombre de réponses utiles.

Une exactitude se lit enfin avec sa marge. Un même candidat interrogé sur le
même service peut ne pas rendre les mêmes réponses d'une passe à l'autre : les
sorties successives sont consignées dans le manifeste. La dispersion des
résultats peut être comparée à l'écart entre le meilleur thème et le thème le
plus faible ; le verdict explicite si cette variation change le franchissement
des seuils. Le total de jetons, lui, compte la longueur des invites et des
réponses, pas leur justesse.

In [9]:
'''Le verdict : la comparaison aux seuils, seuil par seuil.

Un seuil franchi est une condition remplie ; un seuil manque est une condition
non remplie. Le verdict global est l'agregat : il ne passe que si tous les
seuils sont franchis.
'''

# Chaque controle porte son format d'affichage : exactitudes en quatre
# decimales, latence en deux, volumes de jetons en un.
CONTROLES = [
    ('exactitude globale', METRIQUES['exactitude_globale'], '>=',
     SEUILS['exactitude_globale_min'], '%.4f'),
    ('exactitude par theme (la plus faible)',
     min((v['exactitude'] for v in METRIQUES_PAR_THEME.values()), default=0.0), '>=',
     SEUILS['exactitude_par_theme_min'], '%.4f'),
    ('taux d abstention', METRIQUES['taux_abstention'], '<=',
     SEUILS['taux_abstention_max'], '%.4f'),
    ('latence p95 (s)', METRIQUES['latence_p95_s'], '<=',
     SEUILS['latence_p95_s_max'], '%.2f'),
    ('jetons par requete', METRIQUES['jetons_par_requete'], '<=',
     SEUILS['jetons_par_requete_max'], '%.1f'),
    ('jetons par bonne reponse', METRIQUES['jetons_par_bonne_reponse'], '<=',
     SEUILS['jetons_par_bonne_reponse_max'], '%.1f'),
]

SEUILS_FRANCHIS: list[str] = []
SEUILS_NON_FRANCHIS: list[str] = []
for nom, valeur, comparateur, seuil, gabarit in CONTROLES:
    franchi = valeur >= seuil if comparateur == '>=' else valeur <= seuil
    ligne = nom + ' : ' + (gabarit % valeur) + ' ' + comparateur + ' ' + str(seuil)
    (SEUILS_FRANCHIS if franchi else SEUILS_NON_FRANCHIS).append(ligne)

print('=== Verdict du protocole ===')
for ligne in SEUILS_FRANCHIS:
    print('  franchi    ', ligne)
for ligne in SEUILS_NON_FRANCHIS:
    print('  non franchi', ligne)
print('seuils franchis : %d / %d' % (len(SEUILS_FRANCHIS), len(CONTROLES)))
VERDICT = 'PASS' if not SEUILS_NON_FRANCHIS else 'FAIL'
print('VERDICT :', VERDICT)


=== Verdict du protocole ===
  franchi     exactitude globale : 0.6370 >= 0.5
  franchi     taux d abstention : 0.0000 <= 0.1
  franchi     latence p95 (s) : 0.64 <= 30.0
  franchi     jetons par requete : 152.9 <= 600
  franchi     jetons par bonne reponse : 240.0 <= 4000
  non franchi exactitude par theme (la plus faible) : 0.3125 >= 0.35
seuils franchis : 5 / 6
VERDICT : FAIL


### Lecture du résultat

Le verdict est l'agrégat des seuils : un seul seuil manque et le protocole vaut
`FAIL`, même si l'exactitude globale est bonne. C'est le but — un protocole qui
ne juge qu'un seul critère n'en est pas un.

Ici, le seuil de qualité thématique est le seul seuil non franchi. La sortie du
code donne la valeur mesurée et la compare au plancher déclaré ; il n'est pas
nécessaire de recopier ici cette mesure, car elle dépend de la passe. Le verdict
global reste `FAIL` tant qu'un seuil est manqué.

La lecture du verdict se fait ligne à ligne :

- une ligne **franchi** est une condition remplie par la mesure ;
- une ligne **non franchi** est une condition à traiter : soit le seuil est mal
  calibré, soit le candidat ne tient pas la promesse, soit la mesure est en
  cause.

Un seuil franchi n'est pas non plus une garantie : il dit que la mesure a
atteint la valeur convenue, pas que le système est bon. Et un seuil manqué ne
dit pas davantage : il dit que ce qui est mesuré ne tient pas la promesse, et
que la suite du travail est de trancher entre les trois causes ci-dessus.

In [10]:
'''Ou le protocole echoue : les erreurs, groupees par theme.

Un verdict sans analyse d echec est un verdict que l on ne peut pas corriger.
'''


def lettres_dans_la_portee(trace: dict) -> list[str]:
    '''Les lettres isolees d une reponse qui designent une option de sa question.'''
    borne = chr(ord('A') + trace['nb_options'])
    return [lettre for lettre in lettres_isolees(trace['reponse']) if lettre < borne]


# On ne remonte que les lettres rendues, jamais le texte brut de la reponse :
# le rapport reste lisible, et rien n y laisse passer un echo du candidat.
print('=== Echecs par theme ===')
for theme in sorted(METRIQUES_PAR_THEME):
    echecs = [r for r in RESULTATS if r['theme'] == theme and r['statut'] != 'correct']
    print('  -', theme, ':', len(echecs), 'reponse(s) non correcte(s) sur',
          METRIQUES_PAR_THEME[theme]['questions'])
    for trace in echecs[:5]:
        print('      ', trace['id'], '->', trace['statut'],
              '| lettre rendue :', extraire_lettre(trace['reponse']))
print()
# La regle du juge retient la premiere lettre isolee. Ce qu elle a a trancher
# se mesure : une reponse qui porte plusieurs lettres isolees distinctes, dans
# la plage des options, est ambigue pour un juge a lettre.
AMBIGUES = [
    (trace['id'], lettres_dans_la_portee(trace))
    for trace in RESULTATS
    if len(lettres_dans_la_portee(trace)) > 1
]
print('=== Lettres rendues ===')
print('reponses portant plusieurs lettres isolees dans la plage des options :', len(AMBIGUES))
for identifiant, lettres in AMBIGUES[:10]:
    print('      ', identifiant, '->', ' '.join(lettres))
print('questions ecartees du jeu de reference :', len(EXCLUSIONS))
for identifiant, motif in EXCLUSIONS:
    print('  -', identifiant, ':', motif)


=== Echecs par theme ===
  - dl-evaluation-seance : 6 reponse(s) non correcte(s) sur 32
       dl-001 -> incorrect | lettre rendue : A
       dl-004 -> incorrect | lettre rendue : A
       dl-006 -> incorrect | lettre rendue : B
       dl-015 -> incorrect | lettre rendue : A
       dl-022 -> incorrect | lettre rendue : B
  - ia-1-introduction-agents : 3 reponse(s) non correcte(s) sur 13
       ia1-004 -> incorrect | lettre rendue : C
       ia1-006 -> incorrect | lettre rendue : A
       ia1-014 -> incorrect | lettre rendue : H
  - ia-2-resolution-problemes : 22 reponse(s) non correcte(s) sur 32
       ia2-006 -> incorrect | lettre rendue : D
       ia2-008 -> incorrect | lettre rendue : H
       ia2-009 -> incorrect | lettre rendue : B
       ia2-011 -> incorrect | lettre rendue : A
       ia2-012 -> incorrect | lettre rendue : E
  - ia-3-logique-bases-connaissances : 5 reponse(s) non correcte(s) sur 16
       ia3-001 -> incorrect | lettre rendue : C
       ia3-008 -> incorrect | lett

### Lecture du résultat

Le détail des échecs dit où porter l'effort. La sortie montre la répartition des
réponses non correctes par thème ; le thème dont l'exactitude est la plus basse
porte aussi le seuil manqué du verdict.

Le hasard de référence donne un second repère, calculé à partir du nombre
d'options proposées pour chaque question. Le seuil manqué est un seuil de
**qualité**, pas un seuil de devinette : un score se lit face à ce repère.

Le détail affiche aussi ce que la règle du juge a eu à trancher. Il compte les
réponses qui portent plusieurs lettres isolées dans la plage des options :
aucun verdict ne dépend de l'ordre des lettres si ce nombre est nul.

Deux lectures restent possibles, et le protocole ne tranche pas entre elles à
lui seul : soit le candidat peine sur cette famille de questions — énoncés
longs, plusieurs options —, soit les clés de correction y sont plus strictes
que le reste de la banque. Vérifier demande de lire les questions, pas de
refaire la mesure.

La liste des questions écartées est rappelée à la fin : elle fait partie du
résultat. Un rapport qui n'affiche pas ses exclusions laisse croire que la
mesure porte sur la banque entière.

## Exercices

Trois exercices, du plus simple au plus complet : compter, agréger, décider.

### Exercice 1 — compter par thème

Le rapport a besoin d'une répartition par thème. Écrivez la fonction qui la
produit à partir d'une liste de questions.

- `# Etape 1` : parcourez la liste des questions.
- `# Etape 2` : pour chaque question, ajoutez une unité au compte de son thème.
- `# Indice` : un dictionnaire vide et une boucle ; le module `collections` fournit `Counter`, qui fait déjà le travail.


In [11]:
# TODO etudiant : implementez le corps de la fonction.
# Etape 1 : parcourez la liste des questions.
# Etape 2 : ajoutez une unite au compte du theme de chaque question.
# Indice : un dictionnaire vide et une boucle ; `collections.Counter` fait deja le travail.


def compte_par_theme(questions: list[dict]) -> dict[str, int]:
    '''Compte les questions par theme et rend le nombre de questions de chaque theme.'''
    return None


repartition = compte_par_theme(JEU_REFERENCE)
print('Exercice a completer')
if repartition:
    for theme, effectif in sorted(repartition.items()):
        print('  -', theme, ':', effectif)
else:
    print('  (en attente de votre implementation)')


Exercice a completer
  (en attente de votre implementation)


### Exercice 2 — agréger les résultats par thème

Les traces de la passe portent chacune un thème et un statut. Écrivez la
fonction qui rend, pour chaque thème, l'exactitude obtenue.

- `# Etape 1` : regroupez les traces par thème.
- `# Etape 2` : pour chaque groupe, comptez les statuts `correct` et divisez par la taille du groupe.
- `# Indice` : un dictionnaire de dictionnaires ; attention à ne pas diviser par un groupe vide.


In [12]:
# TODO etudiant : implementez le corps de la fonction.
# Etape 1 : regroupez les traces par theme.
# Etape 2 : comptez les statuts `correct` et divisez par la taille du groupe.
# Indice : un dictionnaire de dictionnaires ; un groupe vide ne se divise pas.


def exactitude_par_theme(resultats: list[dict]) -> dict[str, float]:
    '''Rend l'exactitude du candidat, par theme.'''
    return None


par_theme = exactitude_par_theme(RESULTATS)
print('Exercice a completer')
if par_theme:
    for theme, exactitude in sorted(par_theme.items()):
        print('  -', theme, ': %.4f' % exactitude)
else:
    print('  (en attente de votre implementation)')


Exercice a completer
  (en attente de votre implementation)


### Exercice 3 — décider

Le verdict est une comparaison aux seuils. Écrivez la fonction qui rend le
verdict global à partir des métriques et des seuils.

- `# Etape 1` : comparez chaque métrique au seuil qui lui correspond.
- `# Etape 2` : le verdict global vaut `PASS` si aucun seuil n'est manqué, `FAIL` sinon.
- `# Indice` : un seuil se franchit quand la valeur dépasse le seuil pour un `>=`, et quand elle reste en dessous pour un `<=` ; un seul seuil manqué suffit à faire échouer l'ensemble.


In [13]:
# TODO etudiant : implementez le corps de la fonction.
# Etape 1 : comparez chaque metrique au seuil qui lui correspond.
# Etape 2 : le verdict global vaut PASS si aucun seuil n'est manque, FAIL sinon.
# Indice : un seuil se franchit quand la valeur depasse le seuil pour un >=, et quand elle reste en dessous pour un <=.


def verdict_seuils(metriques: dict, seuils: dict) -> str:
    '''Rend le verdict global : PASS si tous les seuils sont franchis, FAIL sinon.'''
    return None


verdict = verdict_seuils(METRIQUES, SEUILS)
print('Exercice a completer')
print('  verdict calcule :', verdict if verdict else '(en attente de votre implementation)')


Exercice a completer
  verdict calcule : (en attente de votre implementation)


## À retenir

- **Évaluer, c'est d'abord déclarer.** Route, référence, juge, seuils : ce qui est écrit avant la mesure est une décision ; ce qui est écrit après est un récit.
- **Le juge fait la mesure.** Face à un juge probabiliste, on ne sait plus si une erreur vient du candidat ou du juge. Un juge déterministe, vérifié par auto-tests, rend l'erreur attribuable.
- **Une exclusion affichée vaut mieux qu'une exclusion silencieuse.** Écarter une question non notable et le dire protège l'interprétation du chiffre.
- **Le coût se compte en réponses, pas en appels.** Une requête qui ne produit aucune réponse est une dépense sans retour.
- **Un verdict est une comparaison.** `PASS` et `FAIL` se lisent par rapport à des seuils écrits ; ils ne sont jamais une impression.

## Vérifiez votre compréhension

1. Pourquoi le juge classe-t-il une lettre hors de la plage des options en abstention plutôt qu'en faute ? *(Réponse repliée ci-dessous.)*
2. Que devient le verdict si le seuil d'exactitude est franchi mais que le taux d'abstention dépasse sa limite ?
3. Pourquoi les seuils sont-ils écrits avant la passe de mesure ?

<details><summary>Réponse 1</summary>
Une lettre hors de la plage des options n'identifie aucune proposition : ce n'est
pas une réponse fausse, c'est une réponse inexploitable. La compter comme faute
ferait baisser l'exactitude pour une raison qui ne concerne pas le candidat. La
compter comme abstention la sort du dénominateur des réponses exploitables, ce
qui est sa vraie place.
</details>

<details><summary>Réponse 2</summary>
Le verdict global est l'agrégat des seuils : un seul seuil manqué et le
protocole vaut `FAIL`, même si l'exactitude est bonne. C'est le but : un
protocole qui ne juge qu'un seul critère n'en est pas un.
</details>

<details><summary>Réponse 3</summary>
Parce qu'un seuil écrit après la mesure ne décide plus : il décrit ce qui s'est
produit. La valeur d'un seuil est de contraindre une décision à l'avance ; sans
cela, n'importe quel résultat se déclare conforme.
</details>

## Pour aller plus loin

- Le carnet [`13b_Agent_Evaluation.ipynb`](../../GenAI/Texte/13b_Agent_Evaluation.ipynb) pose le même protocole pour un système agentique : on y ajoute un juge LLM avec garde-fou anti-bias, absent d'ici.
- La banque de questions publiée se lit seule : [`qcm/README.md`](../qcm/README.md) en donne le format et la provenance.
- Le manifeste du projet, [`evaluation-livraison.json`](evaluation-livraison.json), reprend ce protocole sous la forme des parcours verticaux : référence, juge, seuils et verdict.
- La séparation entre la référence, le juge et le rapport est la méthode classique de l'évaluation comparative de systèmes : elle se retrouve dans les travaux d'évaluation holistique des modèles.
